In [3]:
# explore the dayzer prediction on flow ratio to the actual constraint that's gonna happen 
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from nighthawk.util import bigquery_functions
from nighthawk.util.sql_functions import download_df_from_sql_db

def bq(q):
    return bigquery_functions.download_df_from_bq(q)

def check(name, fn):
    print(f"\n{'='*70}\n[{name}]")
    try:
        fn()
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {str(e)[:300]}")

# 1. dayzer flow for SPP
def c1():
    df = bq("SELECT COUNT(*) AS n, MIN(DATE(Date)) AS min_dt, MAX(DATE(Date)) AS max_dt "
            "FROM `movetocloud-999.dayzer.resultConstraintVE` WHERE marketName='SPP'")
    print("  resultConstraintVE SPP:", df.to_dict("records"))
check("1. dayzer.resultConstraintVE SPP rows", c1)

# 2. dayzer ISOName for SPP
def c2():
    df = bq("SELECT COUNT(*) AS n, COUNT(DISTINCT CID) AS cids "
            "FROM `movetocloud-999.dayzer.ISOName` WHERE marketName='SPP' AND CID>0")
    print("  ISOName SPP:", df.to_dict("records"))
check("2. dayzer.ISOName SPP", c2)

# 3. SPP internal constraint master (MySQL)
def c3():
    for tbl in ["spp_constraints.ConstraintMaster_v3", "spp_constraints.ConstraintFamilyMaster",
                "spp_constraints.MonitoredMaster_v3"]:
        try:
            df = download_df_from_sql_db(f"SELECT COUNT(*) AS n FROM {tbl}")
            print(f"  {tbl}: n={df['n'].iloc[0]}")
        except Exception as e:
            print(f"  {tbl}: MISSING ({str(e)[:80]})")
check("3. SPP constraint master (MySQL)", c3)

# 4. SPP rep constraint tables
def c4():
    for tbl in ["test.SPP_DA_RepConstraintMaster", "test.SPP_RT_RepConstraintMaster"]:
        try:
            df = download_df_from_sql_db(f"SELECT COUNT(*) AS n FROM {tbl}")
            print(f"  {tbl}: n={df['n'].iloc[0]}")
        except Exception as e:
            print(f"  {tbl}: MISSING ({str(e)[:80]})")
check("4. SPP rep-constraint tables (MySQL)", c4)

# 5. SPP DART mvalue via ConstraintFamily.get_mvalues
def c5():
    from nighthawk.data.network.constraint_family import ConstraintFamily
    cf = ConstraintFamily(opexchange="SPP", constraint_family_num_df=None)
    da = cf.get_mvalues(start_dt="2026-05-25", end_dt="2026-06-05", type="DA", level="constraint_family")
    print("  SPP DA mvalue rows:", len(da), "| cols:", list(da.columns)[:8])
    if len(da):
        print("  date range:", da['dt'].min(), "->", da['dt'].max())
check("5. SPP DART mvalue (ConstraintFamily.get_mvalues)", c5)

print("\nDONE")



[1. dayzer.resultConstraintVE SPP rows]
  resultConstraintVE SPP: [{'n': 376185864.0, 'min_dt': '2020-02-25', 'max_dt': '2026-06-09'}]

[2. dayzer.ISOName SPP]
  ISOName SPP: [{'n': 174403, 'cids': 2623}]

[3. SPP constraint master (MySQL)]
  spp_constraints.ConstraintMaster_v3: n=19537
  spp_constraints.ConstraintFamilyMaster: n=5817
  spp_constraints.MonitoredMaster_v3: n=4516

[4. SPP rep-constraint tables (MySQL)]
  test.SPP_DA_RepConstraintMaster: n=517
  test.SPP_RT_RepConstraintMaster: n=668

[5. SPP DART mvalue (ConstraintFamily.get_mvalues)]
  FAILED: TypeError: ConstraintFamily.get_mvalues() got an unexpected keyword argument 'level'

DONE


In [1]:
# ============================================================================
# SPP constraint-family exposure build — fully inline (no prod-pipeline edits)
# Writes only to temp.* / *_SPP_test tables. Reads prod/dayzer/MySQL read-only.
# ============================================================================
import sys, warnings, re
warnings.filterwarnings("ignore")
sys.path.append("/var/www/python/Prod/nighthawk/")
import pandas as pd, numpy as np
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from datetime import datetime, timedelta
import pytz
from nighthawk.util import bigquery_functions
from nighthawk.util.sql_functions import download_df_from_sql_db
from nighthawk.data.network.constraint import Constraint
from nighthawk.data.network.constraint_family import ConstraintFamily

OPEX = "SPP"
SUFFIX = "_SPP_test"          # all SPP component tables get this suffix
DATASET = "constraint_family_exposure"
ICE_LABEL = "INDIANAHUB"      # SPP uses INDIANAHUB ice price as proxy (per decision)

bid_dt = (datetime.now(pytz.timezone("America/Chicago")) + timedelta(days=1)).strftime("%Y-%m-%d")
# bid_dt = "2026-06-09"
start_dt = (pd.to_datetime(bid_dt) - pd.Timedelta("1105 day")).strftime("%Y-%m-%d")  # ~3y lookback
end_dt   = bid_dt
mvalue_threshold = 100

print("bid_dt     :", bid_dt)
print("lookback   :", start_dt, "->", end_dt)
print("suffix     :", SUFFIX)


bid_dt     : 2026-06-10
lookback   : 2023-06-01 -> 2026-06-10
suffix     : _SPP_test


# Check for the dayzer flow ratio prediction vs next day actual DART value change. 

In [2]:
# --- Step A: dayzer ConstraintId -> internal oops_constraint_num mapping (SPP) ---
# SPP dayzer names embed the readable monitored line after a colon, e.g.
#   "8765B_LN@HAYMAKR4<13386>:LN HAYMAKR4 - CIMARRON"
# We parse that segment and normalize to match internal monitored_clean.
# Contingency names don't share a convention, so we match MONITORED-ONLY
# (covers ~92% of dayzer SPP CIDs). Approximate — flagged.

dz = bigquery_functions.download_df_from_bq(f"""
    SELECT DISTINCT CID AS ConstraintId, ISOConstraint AS monitored_name, ISOContingency AS contingency_name
    FROM `movetocloud-999.dayzer.ISOName`
    WHERE marketName='{OPEX}' AND CID > 0
""")
internal_da = Constraint(market=OPEX).get_constraint_details(da_or_rt="DA")[
    ["oops_constraint_num", "monitored_clean", "contingency_clean"]].rename(
    columns={"monitored_clean": "monitored_name", "contingency_clean": "contingency_name"})

def _spp_mon_key(s):
    s = str(s)
    if ":" in s:                       # descriptive segment after first colon
        s = s.split(":", 1)[1]
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")

dz["monitored_key"] = dz["monitored_name"].map(_spp_mon_key)
internal_da["monitored_key"] = internal_da["monitored_name"].map(_spp_mon_key)
internal_rep = internal_da.drop_duplicates(subset="monitored_key")   # one internal con per monitored key

dayzer_oops_mapping = pd.merge(
    internal_rep[["oops_constraint_num", "monitored_name", "contingency_name", "monitored_key"]],
    dz[["ConstraintId", "monitored_key"]], on="monitored_key", how="inner"
).drop(columns="monitored_key").rename(columns={"oops_constraint_num": "oops_constraint_num_da"})

dayzer_oops_mapping_bq = bigquery_functions.upload_to_bq_from_dataframe(
    dayzer_oops_mapping, "temp", f"{OPEX}_dayzer_oops_mapping{SUFFIX}", temp=True)

print("dayzer SPP CIDs total      :", dz["ConstraintId"].nunique())
print("mapped CIDs                :", dayzer_oops_mapping["ConstraintId"].nunique(),
      f"({100*dayzer_oops_mapping['ConstraintId'].nunique()/dz['ConstraintId'].nunique():.0f}%)")
print("mapping table              :", dayzer_oops_mapping_bq)
display(dayzer_oops_mapping.head())


dayzer SPP CIDs total      : 2623
mapped CIDs                : 2412 (92%)
mapping table              : temp.SPP_dayzer_oops_mapping_SPP_test


,oops_constraint_num_da,monitored_name,contingency_name,ConstraintId
0,151242,ln-texas_co,base,1000070
1,683,ln134pentp-mcclai,lnmcclai-saraa138okge,3020455
2,326621,ln134pentp-sgate,okge:mcclaiplval:138:1:9,3022434
3,57407,ln134pentp-westmor4,base,3020608
4,57407,ln134pentp-westmor4,base,5000871


In [3]:
# --- Step B: internal oops_constraint_num -> constraint_family_num mapping (SPP) ---
oops_family = download_df_from_sql_db("""
    SELECT constraintNum AS oops_constraint_num, constraintFamilyNum AS constraint_family_num
    FROM spp_constraints.ConstraintFamilyMaster
    WHERE familyTypeNum = 1
""").drop_duplicates()

oops_family_bq = bigquery_functions.upload_to_bq_from_dataframe(
    oops_family, "temp", f"{OPEX}_oops_constraint_family_mapping{SUFFIX}", temp=True)
print("oops->family rows:", len(oops_family), "| families:", oops_family["constraint_family_num"].nunique())
print("table:", oops_family_bq)
display(oops_family.head())


oops->family rows: 2903 | families: 1257
table: temp.SPP_oops_constraint_family_mapping_SPP_test


,oops_constraint_num,constraint_family_num
0,7,6
1,1216,6
2,1580,6
3,2011,6
4,11,10


In [7]:
# --- Step C: dayzer FlowRatio per constraint_family per day (SPP) ---
# Mirrors the MISO/PJM daily block: aggregate to worst-hour FlowRatio per constraint,
# then pick the top-FlowRatio constraint (rank=1) per (dt, family).
flow_query = f"""
SELECT CAST(dt AS STRING) AS dt, constraint_family_num, ShadowPrice, Flows, FlowRatio, MinFlowLimit, MaxFlowLimit
FROM (
  SELECT dt, ConstraintId, oops_constraint_num_da, constraint_family_num,
         SUM(ShadowPrice) AS ShadowPrice, MAX(Flows) AS Flows,
         MAX(MinFlowLimit) AS MinFlowLimit, MIN(MaxFlowLimit) AS MaxFlowLimit,
         MAX(CASE WHEN Flows < 0 THEN Flows/NULLIF(MinFlowLimit,0) ELSE Flows/NULLIF(MaxFlowLimit,0) END) AS FlowRatio,
         ROW_NUMBER() OVER (PARTITION BY dt, constraint_family_num ORDER BY
             MAX(CASE WHEN Flows < 0 THEN Flows/NULLIF(MinFlowLimit,0) ELSE Flows/NULLIF(MaxFlowLimit,0) END) DESC) AS rnk
  FROM (
    SELECT CAST(Date AS DATE) AS dt, CAST(Hour AS INT) AS hr, a.ConstraintId,
           b.oops_constraint_num_da, c.constraint_family_num,
           ShadowPrice, Flows, MinFlowLimit, MaxFlowLimit
    FROM `movetocloud-999.dayzer.resultConstraintVE` AS a
    INNER JOIN `movetocloud-999.{dayzer_oops_mapping_bq}` AS b ON a.ConstraintId = b.ConstraintId
    INNER JOIN `movetocloud-999.{oops_family_bq}`         AS c ON b.oops_constraint_num_da = c.oops_constraint_num
    WHERE DATE(Date) BETWEEN DATE('{start_dt}') AND DATE('{end_dt}')
      AND marketName = '{OPEX}' AND ScenarioId = 1
  )
  GROUP BY dt, ConstraintId, oops_constraint_num_da, constraint_family_num
)
WHERE rnk = 1
ORDER BY constraint_family_num, dt
"""
dayzer_table_bq = bigquery_functions.create_temp_table_from_query(
    flow_query, "temp", f"{OPEX}_dayzer_table{SUFFIX}", temp=True)
print("dayzer_table:", dayzer_table_bq)

q = f"SELECT COUNT(*) n, COUNT(DISTINCT constraint_family_num) fams, MIN(dt) mn, MAX(dt) mx FROM `movetocloud-999.{dayzer_table_bq}`"
print(bigquery_functions.download_df_from_bq(q).to_dict("records"))



dayzer_table: temp.SPP_dayzer_table_SPP_test
[{'n': 222902, 'fams': 205, 'mn': '2023-06-01', 'mx': '2026-06-10'}]


In [10]:
display(bigquery_functions.download_df_from_bq(
    f"SELECT * FROM `movetocloud-999.{dayzer_table_bq}` WHERE dt='{bid_dt}' ORDER BY FlowRatio DESC LIMIT 20 "))

,dt,constraint_family_num,ShadowPrice,Flows,FlowRatio,MinFlowLimit,MaxFlowLimit
0,2026-06-10,11,-2500.000000,232.18700,1.156886,-200.70000,200.70000
1,2026-06-10,140,-500.000000,325.17352,1.156378,-281.20001,281.20001
2,2026-06-10,93,-283.954219,140.73285,1.042466,-135.00000,135.00000
3,2026-06-10,115,-200.000000,325.17352,1.000534,-325.00000,325.00000
4,2026-06-10,432,-3980.267956,178.24757,1.000267,-178.20000,178.20000
5,2026-06-10,75,-24.711369,153.00000,1.000000,-153.00000,153.00000
6,2026-06-10,230,-763.271947,80.00000,1.000000,-80.00000,80.00000
7,2026-06-10,119,-822.134491,171.89999,1.000000,-171.89999,171.89999
8,2026-06-10,380,-420.528180,95.00000,1.000000,-95.00000,95.00000
9,2026-06-10,185,-1251.646439,171.89999,1.000000,-171.89999,171.89999


In [11]:
# --- Step D: DART mvalue per family/day (SPP) + mvalue cut for families of interest ---
cf = ConstraintFamily(opexchange=OPEX, constraint_family_num_df=None)
da = cf.get_mvalues(start_dt=start_dt, end_dt=end_dt, type="DA", granularity="daily").rename(
    columns={"mvalue": "dam", "constraintFamilyNum": "constraint_family_num"})
rt = cf.get_mvalues(start_dt=start_dt, end_dt=end_dt, type="RT", granularity="daily").rename(
    columns={"mvalue": "rtm", "constraintFamilyNum": "constraint_family_num"})

dartm = pd.merge(da[["constraint_family_num", "dt", "dam"]],
                 rt[["constraint_family_num", "dt", "rtm"]],
                 on=["constraint_family_num", "dt"], how="outer")
dartm["dt"] = pd.to_datetime(dartm["dt"]).dt.strftime("%Y-%m-%d")
dartm[["dam", "rtm"]] = dartm[["dam", "rtm"]].fillna(0)

# mvalue cut: keep families that ever exceed the threshold in DA or RT
fam_max = dartm.groupby("constraint_family_num").agg(damx=("dam", "max"), rtmx=("rtm", "max")).reset_index()
interested = fam_max[(fam_max["damx"] > mvalue_threshold) | (fam_max["rtmx"] > mvalue_threshold)]["constraint_family_num"]
dartm = dartm[dartm["constraint_family_num"].isin(interested)].reset_index(drop=True)

print("families total:", fam_max.shape[0], "| above mvalue cut:", len(interested))
print("dartm rows:", len(dartm), "| date range:", dartm["dt"].min(), "->", dartm["dt"].max())
display(dartm.sort_values("dam", ascending=False).head())


families total: 2177 | above mvalue cut: 0
dartm rows: 0 | date range: nan -> nan


,constraint_family_num,dt,dam,rtm


NameError: name 'dartm' is not defined

In [ ]:
# --- Step E: INDIANAHUB ice price (proxy) + normalized mvalues ---
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh
ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
    [1656], "MISO", [ICE_LABEL], start_dt, end_dt, var_spec=["f"], impute=True)
ice_df = ice_df.rename(columns={f"{ICE_LABEL}_ice_elec_price_forecast_f": "ice_price"})
ice_df = ice_df.groupby("dt").agg({"ice_price": "mean"}).reset_index()
ice_df["dt"] = pd.to_datetime(ice_df["dt"]).dt.strftime("%Y-%m-%d")

dartm = dartm.merge(ice_df, on="dt", how="left")
dartm["ice_price"] = dartm["ice_price"].replace(0, np.nan).fillna(method="ffill").fillna(method="bfill")
dartm["dam_norm"] = dartm["dam"] / dartm["ice_price"]
dartm["rtm_norm"] = dartm["rtm"] / dartm["ice_price"]
print("ice_price range:", round(dartm["ice_price"].min(), 2), "->", round(dartm["ice_price"].max(), 2),
      "| nulls:", dartm["ice_price"].isna().sum())
display(dartm.head())


In [ ]:
# --- Step F: lagged rolling price-derived stats per family (avoid look-ahead) ---
# DA lagged 1 day, RT lagged 2 days (mirrors pipeline dam_minus_1 / rtm_minus_2).
dartm = dartm.sort_values(["constraint_family_num", "dt"]).reset_index(drop=True)
g = dartm.groupby("constraint_family_num")

dartm["dam_l1"]      = g["dam"].shift(1)
dartm["rtm_l2"]      = g["rtm"].shift(2)
dartm["dam_norm_l1"] = g["dam_norm"].shift(1)

def roll(col, win, fn):
    return dartm.groupby("constraint_family_num")[col].transform(
        lambda s: getattr(s.rolling(win, min_periods=1), fn)())

dartm["da_max_in7d"]      = roll("dam_l1", 7, "max")
dartm["rt_max_in7d"]      = roll("rtm_l2", 7, "max")
dartm["da_max_in30d"]     = roll("dam_l1", 30, "max")
dartm["rt_max_in30d"]     = roll("rtm_l2", 30, "max")
dartm["da_avg_in30d"]     = roll("dam_l1", 30, "mean")
dartm["da_norm_avg_in5d"] = roll("dam_norm_l1", 5, "mean")
dartm["da_rt_norm_max"]   = dartm[["dam_norm", "rtm_norm"]].max(axis=1)

print("rolling cols added. bid_dt sample:")
display(dartm[dartm["dt"] == bid_dt][
    ["constraint_family_num", "dam", "rtm", "rt_max_in7d", "da_max_in7d",
     "da_avg_in30d", "da_max_in30d", "rt_max_in30d", "da_norm_avg_in5d"]].head())


In [ ]:
# --- Step G: KV per family (rep constraint -> monitored equipment -> Powerflow KV) ---
from nighthawk.data.network.powerflow import Powerflow

rep_con = cf.get_rep_constraints().rename(columns={"constraintFamilyNum": "constraint_family_num"})
rep_con = rep_con[rep_con["constraint_family_num"].isin(interested)].copy()
rep_con["oops_constraint_num"] = rep_con["repConstraintNum"].astype(int)

c_obj = Constraint(pd.DataFrame({"oops_constraint_num": rep_con["oops_constraint_num"].unique().tolist()}), OPEX)
eq = c_obj.get_monitored_equipment_details()
eq = eq.loc[eq.groupby("oops_constraint_num")["monitored_eqNum"].idxmax()]   # one eq per constraint

pf = Powerflow(OPEX)
kv_df = pf.get_eq_name_and_kv_for_frontend(eq["monitored_eqNum"].unique().tolist())

feat = rep_con[["constraint_family_num", "oops_constraint_num"]].merge(
    eq[["oops_constraint_num", "monitored_eqNum"]].rename(columns={"monitored_eqNum": "eqNum"}),
    on="oops_constraint_num", how="left").merge(
    kv_df[["eqNum", "KV"]], on="eqNum", how="left")
feat = feat.drop_duplicates(subset="constraint_family_num")

print("families with KV:", feat["KV"].notna().sum(), "/", len(feat))
display(feat.head())


In [ ]:
# --- Step H: assemble stats (history) = price stats + FlowRatio + KV + KV_group ---
dayzer_df = bigquery_functions.download_df_from_bq(
    f"SELECT CAST(dt AS STRING) AS dt, constraint_family_num, FlowRatio FROM `movetocloud-999.{dayzer_table_bq}`")
dayzer_df["dt"] = pd.to_datetime(dayzer_df["dt"]).dt.strftime("%Y-%m-%d")

stats = dartm.merge(dayzer_df, on=["dt", "constraint_family_num"], how="left")
stats = stats.merge(feat[["constraint_family_num", "KV"]], on="constraint_family_num", how="left")

stats["FlowRatio"] = pd.to_numeric(stats["FlowRatio"], errors="coerce").fillna(0).clip(upper=1)
stats["KV"] = pd.to_numeric(stats["KV"], errors="coerce")
stats["KV_group"] = pd.cut(
    stats["KV"], bins=[-float("inf"), 1, 115, 138, 345, float("inf")],
    labels=["01_na", "02_<=115", "03_138", "04_345", "05_>=500"]).astype("object").fillna("03_138")

print("stats rows:", len(stats), "| families:", stats["constraint_family_num"].nunique())
print("KV_group counts:\n", stats.drop_duplicates("constraint_family_num")["KV_group"].value_counts())
display(stats[stats["dt"] == bid_dt][
    ["constraint_family_num", "FlowRatio", "KV", "KV_group", "da_rt_norm_max", "da_norm_avg_in5d"]].head())


In [ ]:
# --- Step I: q95 adj-norm tail loss per (family, KV_group) x FlowRatio bucket ---
def quantiles(group):
    fg = group[group["dam"] >= 0]
    def q(lo, hi):
        if hi is None:   m = fg["FlowRatio"] > lo
        elif lo is None: m = fg["FlowRatio"] <= hi
        else:            m = (fg["FlowRatio"] > lo) & (fg["FlowRatio"] <= hi)
        return fg.loc[m, "da_rt_norm_max"].sub(fg["da_norm_avg_in5d"]).clip(lower=0).quantile(0.95)
    return pd.Series({
        "adj_da_rt_norm_max_q95_0_98_1":    q(0.98, None),
        "adj_da_rt_norm_max_q95_0_95_0_98": q(0.95, 0.98),
        "adj_da_rt_norm_max_q95_0_75_0_95": q(0.75, 0.95),
        "adj_da_rt_norm_max_q95_0_0_75":    q(None, 0.75),
    })

qres = stats.groupby(["constraint_family_num", "KV_group"]).apply(quantiles).reset_index().round(2)

# monotonic enforcement: more-binding bucket >= less-binding bucket
qres["adj_da_rt_norm_max_q95_0_95_0_98"] = qres.apply(
    lambda r: r["adj_da_rt_norm_max_q95_0_75_0_95"]
    if pd.isna(r["adj_da_rt_norm_max_q95_0_95_0_98"]) or r["adj_da_rt_norm_max_q95_0_95_0_98"] < r["adj_da_rt_norm_max_q95_0_75_0_95"]
    else r["adj_da_rt_norm_max_q95_0_95_0_98"], axis=1)
qres["adj_da_rt_norm_max_q95_0_98_1"] = qres.apply(
    lambda r: r["adj_da_rt_norm_max_q95_0_95_0_98"]
    if pd.isna(r["adj_da_rt_norm_max_q95_0_98_1"]) or r["adj_da_rt_norm_max_q95_0_98_1"] < r["adj_da_rt_norm_max_q95_0_95_0_98"]
    else r["adj_da_rt_norm_max_q95_0_98_1"], axis=1)

print("quantile rows (family x KV_group):", len(qres))
display(qres.head())


In [ ]:
# --- Step J: TARGET TABLE #2 — SPP ..._with_quantiles (bid_dt) ---
qcols = ["adj_da_rt_norm_max_q95_0_98_1", "adj_da_rt_norm_max_q95_0_95_0_98",
         "adj_da_rt_norm_max_q95_0_75_0_95", "adj_da_rt_norm_max_q95_0_0_75"]

with_quantiles = (stats[stats["dt"] == bid_dt][["dt", "constraint_family_num", "KV_group"]]
                  .merge(qres, on=["constraint_family_num", "KV_group"], how="left")[["dt", "constraint_family_num"] + qcols])

with_quantiles_bq = bigquery_functions.upload_to_bq_from_dataframe(
    with_quantiles, "temp", f"{OPEX}_constraint_family_exposure_with_quantiles{SUFFIX}", temp=True)
print("TARGET #2 with_quantiles:", with_quantiles_bq, "| rows:", len(with_quantiles))
display(with_quantiles.head())


In [ ]:
# --- Step K: SPP portfolio + node-level dfax (per family) ---
# NOTE: get_dfax_on_all_nodes over the interested rep-constraints is the heaviest step.
from nighthawk.data.product import ve as ve_mod
from nighthawk.data.product.ve import DailyBidsManager

portfolio = DailyBidsManager(opexchange=OPEX, bid_date=bid_dt).get_bids_from_table(label="preautomated_cuts")
print("portfolio rows:", len(portfolio), "| strategies:", portfolio["strategy"].unique() if len(portfolio) else "-")

# node-level dfax for the interested families' representative constraints
c_obj_dfax = Constraint(pd.DataFrame({"oops_constraint_num": rep_con["oops_constraint_num"].unique().tolist()}), OPEX)
c_dfax = c_obj_dfax.get_dfax_on_all_nodes(dfax_cutoff=0.01)
ve_obj = ve_mod.VE(OPEX)
biddable = ve_obj.get_biddable_nodes_for_daterange(bid_dt, bid_dt)
c_dfax = c_dfax[c_dfax["node_num"].isin(biddable["node_num"].unique())]

dfax = pd.merge(rep_con[["constraint_family_num", "oops_constraint_num"]], c_dfax, on="oops_constraint_num", how="inner")
# representative (max |dfax|) per (family, node)
idx = dfax.groupby(["constraint_family_num", "node_num"])["dfax"].apply(lambda x: x.abs().idxmax())
dfax = dfax.loc[idx, ["constraint_family_num", "oops_constraint_num", "node_num", "dfax"]].reset_index(drop=True)
print("dfax rows:", len(dfax), "| families:", dfax["constraint_family_num"].nunique(),
      "| nodes:", dfax["node_num"].nunique())
display(dfax.head())


In [ ]:
# --- Step L: TARGET TABLE #1 — SPP constraint_family_exposure (bid_dt) ---
# dfax-weighted short/long exposure per family (mirrors get_portfolio_exposure_for_future_bids).
portfolio["inc_mw"] = np.where(portfolio["incdec"] == "Increment", portfolio["bid_mw"], 0.0)
portfolio["dec_mw"] = np.where(portfolio["incdec"] == "Decrement", portfolio["bid_mw"], 0.0)
bs = portfolio.groupby(["dt", "node_num"], as_index=False).agg(
    bid_mw_inc=("inc_mw", "sum"), bid_mw_dec=("dec_mw", "sum"))
bs["dt"] = pd.to_datetime(bs["dt"]).dt.strftime("%Y-%m-%d")

ex = bs.merge(dfax[["constraint_family_num", "node_num", "dfax"]], on="node_num", how="inner")
ex["short"] = np.where(ex["dfax"] < 0, -ex["dfax"] * ex["bid_mw_dec"],  ex["dfax"] * ex["bid_mw_inc"])
ex["long"]  = np.where(ex["dfax"] < 0, -ex["dfax"] * ex["bid_mw_inc"],  ex["dfax"] * ex["bid_mw_dec"])
short_bid = ex.groupby(["dt", "constraint_family_num"], as_index=False).agg(
    short_bid_mw=("short", "sum"), long_bid_mw=("long", "sum"))

keep = ["dt", "constraint_family_num", "oops_constraint_num", "KV", "KV_group", "FlowRatio",
        "rt_max_in7d", "da_max_in7d", "da_avg_in30d", "da_max_in30d", "rt_max_in30d",
        "dam", "rtm", "dam_norm", "rtm_norm"]
exposure = stats[stats["dt"] == bid_dt].merge(feat[["constraint_family_num", "oops_constraint_num"]],
                                              on="constraint_family_num", how="left")
exposure = exposure[[c for c in keep if c in exposure.columns]].merge(
    short_bid, on=["dt", "constraint_family_num"], how="left")
exposure[["short_bid_mw", "long_bid_mw"]] = exposure[["short_bid_mw", "long_bid_mw"]].fillna(0)

exposure_bq = bigquery_functions.upload_to_bq_from_dataframe(
    exposure, "temp", f"{OPEX}_constraint_family_exposure{SUFFIX}", temp=True)
print("TARGET #1 exposure:", exposure_bq, "| rows:", len(exposure),
      "| families with short>0:", (exposure["short_bid_mw"] > 0).sum())
display(exposure.sort_values("short_bid_mw", ascending=False).head())


In [ ]:
# --- Step M: risk-cut math -> final_factor per family (RC-6 .. RC-9) ---
all_dt = exposure.merge(with_quantiles, on=["dt", "constraint_family_num"], how="inner")
all_dt = all_dt[((all_dt["da_max_in30d"] > 100) | (all_dt["rt_max_in30d"] > 100)) & (all_dt["short_bid_mw"] > 0)].copy()
all_dt = all_dt.merge(ice_df, on="dt", how="left")
all_dt["FlowRatio"] = pd.to_numeric(all_dt["FlowRatio"], errors="coerce").fillna(0).clip(upper=1)

scale = 3
base_limit = 1_000_000 * scale
all_dt["risk_limit"] = base_limit / all_dt["ice_price"]

kv = pd.to_numeric(all_dt["KV"], errors="coerce"); kv_clean = kv.mask(kv <= 0)
all_dt["kv_factor"] = np.select(
    [kv_clean.le(69), kv_clean.le(115), kv_clean.le(161), kv_clean.le(220), kv_clean.le(345), kv_clean.le(500), kv_clean.gt(500)],
    [0.7, 1.1, 1.6, 2.1, 2.6, 3.1, 3.6], default=2.0)
all_dt["risk_limit"] = all_dt["risk_limit"] * all_dt["kv_factor"]

FR, KG = all_dt["FlowRatio"], all_dt["KV_group"]
def band(lo, hi):
    if hi is None: return FR > lo
    if lo is None: return FR <= hi
    return (FR > lo) & (FR <= hi)
floors = {"02_<=115": (46, 26, 20, 1), "03_138": (42, 28, 20, 3), "04_345": (35, 21, 17, 2), "05_>=500": (43, 28, 24, 1)}
rules = []
for kg, (f98, f95, f75, f0) in floors.items():
    rules += [
        (band(0.98, None) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_98_1"].apply(lambda x, f=f98: f if pd.isna(x) or x < f * 0.2 else x)),
        (band(0.95, 0.98) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_95_0_98"].fillna(f95)),
        (band(0.75, 0.95) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_75_0_95"].fillna(f75)),
        (band(None, 0.75) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_0_75"].fillna(f0)),
    ]
all_dt["risk_per_mw"] = np.select([c for c, _ in rules], [v for _, v in rules], default=0.1)
all_dt["risk_per_mw_recent"] = np.where(
    (all_dt["FlowRatio"] > 0.98) | (all_dt["FlowRatio"].isna() & (all_dt["rt_max_in7d"] > 10 * all_dt["da_avg_in30d"]) & (all_dt["rt_max_in7d"] > 400)),
    np.maximum(all_dt["rt_max_in7d"] / all_dt["ice_price"] - all_dt["da_max_in7d"] / all_dt["ice_price"], 0.1), 0.1)

all_dt["long_term_mw_limit"] = all_dt["risk_limit"] / all_dt["risk_per_mw"]
all_dt["recent_mw_limit"]    = all_dt["risk_limit"] / all_dt["risk_per_mw_recent"]
all_dt["KV_mw_limit"]        = 1000 * scale * all_dt["kv_factor"]

nz = all_dt["short_bid_mw"] != 0
for col, lim in [("long_term_risk_scale_factor", "long_term_mw_limit"),
                 ("recent_risk_scale_factor", "recent_mw_limit"),
                 ("kv_mw_scale_factor", "KV_mw_limit")]:
    all_dt[col] = 1.0
    all_dt.loc[nz, col] = np.minimum(all_dt.loc[nz, "short_bid_mw"], all_dt.loc[nz, lim]) / all_dt.loc[nz, "short_bid_mw"]
all_dt["final_factor"] = all_dt[["long_term_risk_scale_factor", "recent_risk_scale_factor", "kv_mw_scale_factor"]].min(axis=1)

print("families considered:", len(all_dt), "| being cut (final_factor<1):", (all_dt["final_factor"] < 1).sum())
display(all_dt[["constraint_family_num", "FlowRatio", "KV_group", "short_bid_mw",
                "risk_per_mw", "long_term_mw_limit", "final_factor"]].sort_values("final_factor").head(15))


In [ ]:
# --- Step N: apply final_factor to portfolio via dfax (RC-11) + summary ---
cut_kv = ["02_<=115", "03_138", "04_345", "05_>=500"]
ff = all_dt[["constraint_family_num", "final_factor"]]
d = dfax.merge(ff, on="constraint_family_num", how="left")
d["final_factor"] = d["final_factor"].fillna(1.0)
# KV_group per family (from exposure); families not cut -> factor stays 1 regardless
d = d.merge(exposure[["constraint_family_num", "KV_group"]].drop_duplicates(), on="constraint_family_num", how="left")
d = d[d["dfax"].abs() > 0.05]
d["inc_factor"] = np.where((d["dfax"] > 0) & (d["KV_group"].isin(cut_kv)), d["final_factor"], 1.0)
d["dec_factor"] = np.where((d["dfax"] < 0) & (d["KV_group"].isin(cut_kv)), d["final_factor"], 1.0)
node_factor = d.groupby("node_num", as_index=False).agg(inc_factor=("inc_factor", "min"), dec_factor=("dec_factor", "min"))

ps = portfolio.merge(node_factor, on="node_num", how="left")
ps["inc_factor"] = ps["inc_factor"].fillna(1.0)
ps["dec_factor"] = ps["dec_factor"].fillna(1.0)
ps["bid_mw_original"] = ps["bid_mw"]
ps["bid_mw"] = np.where(ps["incdec"] == "Increment", ps["bid_mw"] * ps["inc_factor"], ps["bid_mw"] * ps["dec_factor"])

diff = ps.groupby("incdec").agg(orig=("bid_mw_original", "sum"), scaled=("bid_mw", "sum")).reset_index()
diff["reduction_pct"] = (1 - diff["scaled"] / diff["orig"]) * 100
print("=== SPP risk cut — reduction by incdec ===")
display(diff)
print("\nTop affected nodes:")
nd = ps.groupby(["node_num", "incdec"]).agg(orig=("bid_mw_original", "sum"), scaled=("bid_mw", "sum")).reset_index()
nd["reduction"] = nd["orig"] - nd["scaled"]
display(nd.sort_values("reduction", ascending=False).head(15))
